In [21]:
import yaml
import os
import polars as pl
from datetime import datetime 
from functools import partial
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'MMA.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'NMRA.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'output', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SGRX.parquet', 'SKYA.parquet', 'SKYQ.parquet', 'SLND.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TCRT.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XLAB.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [22]:
list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
from datetime import date

tickers_days = {
    "TCRT": ["2026-10-13"],
    "VEEA": ["2026-10-09", "2026-10-05"],
}

columns = ["datetime", "ticker", "open", "high", "low", "close", "volume"]
frames = []

for ticker, days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(f"No existe el paquete Parquet para {ticker}: {parquet_path}")

    requested_days = [date.fromisoformat(day) for day in days]

    ticker_df = (
        pl.read_parquet(parquet_path)
        .filter(pl.col("datetime").dt.date().is_in(requested_days))
        .with_columns(pl.lit(ticker).alias("ticker"))
        .select(columns)
    )

    frames.append(ticker_df)

df = pl.concat(frames).sort(["ticker", "datetime"])
df

datetime,ticker,open,high,low,close,volume
"datetime[μs, UTC]",str,f64,f64,f64,f64,i64
2026-10-13 08:00:00 UTC,"""TCRT""",1.0,1.01,0.9897,0.9897,2177
2026-10-13 08:03:00 UTC,"""TCRT""",0.9905,1.02,0.9896,0.9896,7586
2026-10-13 08:04:00 UTC,"""TCRT""",0.9896,1.0,0.9896,1.0,472
2026-10-13 08:09:00 UTC,"""TCRT""",1.03,1.05,1.03,1.05,1357
2026-10-13 08:14:00 UTC,"""TCRT""",1.0499,1.0499,1.0499,1.0499,363
…,…,…,…,…,…,…
2026-10-09 23:55:00 UTC,"""VEEA""",5.04,5.05,4.99,4.99,5726
2026-10-09 23:56:00 UTC,"""VEEA""",4.98,4.98,4.95,4.95,4462
2026-10-09 23:57:00 UTC,"""VEEA""",4.98,4.98,4.9715,4.98,1456


In [23]:
GROUPS = ["ticker", "trading_day"]


def daily_vwap():
    return [
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over(GROUPS)
            / pl.col("volume").cum_sum().over(GROUPS)
        ).alias("vwap_daily")
    ]


def atr(period=14):
    previous_close = pl.col("close").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    return [
        true_range
        .rolling_mean(window_size=period, min_samples=period)
        .over(GROUPS)
        .alias(f"atr_{period}")
    ]


def rsi(period=14):
    change = pl.col("close").diff().over(GROUPS)
    gain = pl.when(change > 0).then(change).otherwise(0.0)
    loss = pl.when(change < 0).then(-change).otherwise(0.0)

    average_gain = (
        gain.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )
    average_loss = (
        loss.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )

    return [
        pl.when(average_loss == 0)
        .then(100.0)
        .when(average_gain == 0)
        .then(0.0)
        .otherwise(100.0 - (100.0 / (1.0 + average_gain / average_loss)))
        .alias(f"rsi_{period}")
    ]


def donchian_channels(period=20):
    upper = pl.col("high").rolling_max(period, min_samples=period).over(GROUPS)
    lower = pl.col("low").rolling_min(period, min_samples=period).over(GROUPS)

    return [
        upper.alias(f"donchian_upper_{period}"),
        lower.alias(f"donchian_lower_{period}"),
        ((upper + lower) / 2).alias(f"donchian_middle_{period}"),
    ]


def momentum(period=10):
    return [
        (
            pl.col("close")
            - pl.col("close").shift(period).over(GROUPS)
        ).alias(f"momentum_{period}")
    ]

def mean_vol(period=100):
    mean_vol = pl.col('volume').rolling_mean(period,min_samples=period).over(GROUPS)
    return [
        mean_vol.alias(f"mean_vol_{period}")
    ]

def median_vol(period=100):
    mean_vol = pl.col('volume').rolling_median(period,min_samples=period).over(GROUPS)
    return [
        mean_vol.alias(f"median_vol_{period}")
    ]

def ema(period: int = 100):
    return [
        pl.col("close")
        .ewm_mean(span=period, adjust=False, min_samples=period)
        .over(GROUPS)
        .alias(f"ema_{period}")
    ]
def vortex(period: int = 14):
    previous_close = pl.col("close").shift(1).over(GROUPS)
    previous_high = pl.col("high").shift(1).over(GROUPS)
    previous_low = pl.col("low").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    positive_movement = (pl.col("high") - previous_low).abs()
    negative_movement = (pl.col("low") - previous_high).abs()

    true_range_sum = true_range.rolling_sum(period, min_samples=period).over(GROUPS)

    return [
        (
            positive_movement.rolling_sum(period, min_samples=period).over(GROUPS)
            / true_range_sum
        ).alias(f"vortex_plus_{period}"),
        (
            negative_movement.rolling_sum(period, min_samples=period).over(GROUPS)
            / true_range_sum
        ).alias(f"vortex_minus_{period}"),
    ]
def acceleration(period=5, smooth=3, lag=1):
    """Calcula velocidad porcentual suavizada y cambio de velocidad."""
    for name, value in (("period", period), ("smooth", smooth), ("lag", lag)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{name} debe ser un entero positivo")

    previous_close = pl.col("close").shift(period)

    velocity = (
        pl.when(previous_close > 0)
        .then(100.0 * (pl.col("close") / previous_close - 1.0))
        .otherwise(None)
        .ewm_mean(
            span=smooth,
            adjust=False,
            min_samples=smooth,
            ignore_nulls=True,
        )
    )

    return [
        velocity.over(GROUPS).alias(f"velocity_{period}_{smooth}"),
        (velocity - velocity.shift(lag))
        .over(GROUPS)
        .alias(f"acceleration_{period}_{smooth}_{lag}"),
    ]
def williams_r(period=14):
    """Posición del cierre dentro del rango reciente, entre -100 y 0."""
    if isinstance(period, bool) or not isinstance(period, int) or period < 1:
        raise ValueError("period debe ser un entero positivo")

    highest = pl.col("high").rolling_max(period, min_samples=period)
    lowest = pl.col("low").rolling_min(period, min_samples=period)
    price_range = highest - lowest

    return [
        pl.when(price_range > 0)
        .then(-100.0 * (highest - pl.col("close")) / price_range)
        .otherwise(None)
        .over(GROUPS)
        .alias(f"williams_r_{period}")
    ]
def bollinger_bands(
    period=20,
    upper_std=2.0,
    lower_std=2.0,
    ma_type="sma",
):
    """Bandas con media SMA/EMA y multiplicadores independientes."""
    import math

    if isinstance(period, bool) or not isinstance(period, int) or period < 1:
        raise ValueError("period debe ser un entero positivo")

    ma_type = ma_type.lower()
    if ma_type not in ("sma", "ema"):
        raise ValueError("ma_type debe ser 'sma' o 'ema'")

    for name, value in (("upper_std", upper_std), ("lower_std", lower_std)):
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
            or value < 0
        ):
            raise ValueError(f"{name} debe ser un número finito no negativo")

    close = pl.col("close")

    if ma_type == "sma":
        middle = close.rolling_mean(period, min_samples=period)
    else:
        middle = close.ewm_mean(
            span=period,
            adjust=False,
            min_samples=period,
        )

    deviation = close.rolling_std(
        period,
        min_samples=period,
        ddof=0,
    )

    return [
        middle.over(GROUPS).alias(f"bb_middle_{period}"),
        (middle + deviation * upper_std)
        .over(GROUPS)
        .alias(f"bb_upper_{period}"),
        (middle - deviation * lower_std)
        .over(GROUPS)
        .alias(f"bb_lower_{period}"),
    ]
def roc(period=10):
    """Variación porcentual del cierre respecto a n barras atrás."""
    if isinstance(period, bool) or not isinstance(period, int) or period < 1:
        raise ValueError("period debe ser un entero positivo")

    previous_close = pl.col("close").shift(period)

    return [
        pl.when(previous_close > 0)
        .then(100.0 * (pl.col("close") / previous_close - 1.0))
        .otherwise(None)
        .over(GROUPS)
        .alias(f"roc_{period}")
    ]
def add_indicators(frame, indicator_functions):
    expressions = [
        expression
        for indicator_function in indicator_functions
        for expression in indicator_function()
    ]
    return frame.with_columns(expressions)


df_to_plot = (
    df
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort(["ticker", "datetime"])
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),
        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("price"),
    )
)

df_to_plot = add_indicators(
    df_to_plot,
    [
        daily_vwap,
        atr,
        # bollinger_bands,
        # partial(bollinger_bands,50,2),
        rsi,
        # partial(rsi, period=3),  # RSI de 3 períodos
        partial(donchian_channels,20),
        momentum,
        partial(mean_vol,100),
        # median_vol,
        partial(ema, 50),
        # partial(ema, 10),
        # partial(acceleration, period=5, smooth=3, lag=1),
        # vortex,
        # partial(williams_r, period=14),
        partial(roc, period=10),
        partial(bollinger_bands,period=50,ma_type="ema",upper_std=2.0,lower_std=2.0,),
    ],
)

df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,atr_14,rsi_14,donchian_upper_20,donchian_lower_20,donchian_middle_20,momentum_10,mean_vol_100,ema_50,roc_10,bb_middle_50,bb_upper_50,bb_lower_50
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-10-13 04:00:00 EDT,"""TCRT""",1.0,1.01,0.9897,0.9897,2177,2026-10-13,0.996467,0.996467,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:03:00 EDT,"""TCRT""",0.9905,1.02,0.9896,0.9896,7586,2026-10-13,0.999733,0.999005,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:04:00 EDT,"""TCRT""",0.9896,1.0,0.9896,1.0,472,2026-10-13,0.996533,0.998891,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:09:00 EDT,"""TCRT""",1.03,1.05,1.03,1.05,1357,2026-10-13,1.043333,1.004094,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:14:00 EDT,"""TCRT""",1.0499,1.0499,1.0499,1.0499,363,2026-10-13,1.0499,1.005484,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-10-09 19:55:00 EDT,"""VEEA""",5.04,5.05,4.99,4.99,5726,2026-10-09,5.01,5.931664,0.062907,54.661098,5.09,4.9,4.995,0.0354,2312.0,4.914584,0.714488,4.914584,5.080254,4.748915
2026-10-09 19:56:00 EDT,"""VEEA""",4.98,4.98,4.95,4.95,4462,2026-10-09,4.96,5.931612,0.06005,50.272795,5.09,4.9,4.995,-0.05,2346.11,4.915973,-1.0,4.915973,5.08031,4.751636
2026-10-09 19:57:00 EDT,"""VEEA""",4.98,4.98,4.9715,4.98,1456,2026-10-09,4.977167,5.931596,0.058764,53.300911,5.09,4.9,4.995,0.01,2356.4,4.918484,0.201207,4.918484,5.08357,4.753398


In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display


GROUPS = ["ticker", "trading_day"]

plot_df = (
    df_to_plot
    .sort(["ticker", "datetime"])
    .with_row_index("plot_x")
)


def add_segmented_line(fig, frame, column, name, row, color, dash="solid", width=1):
    """Añade una línea sin conectar distintos tickers ni días."""
    show_legend = True

    for segment in frame.partition_by(GROUPS, maintain_order=True):
        valid_segment = segment.drop_nulls(column)

        if valid_segment.height == 0:
            continue

        fig.add_trace(
            go.Scatter(
                x=valid_segment["plot_x"].to_list(),
                y=valid_segment[column].to_list(),
                mode="lines",
                name=name,
                legendgroup=name,
                showlegend=show_legend,
                line={"color": color, "dash": dash, "width": width},
            ),
            row=row,
            col=1,
        )

        show_legend = False


volume_colors = [
    "green" if close > open_price
    else "red" if close < open_price
    else "gray"
    for open_price, close in zip(
        plot_df["open"].to_list(),
        plot_df["close"].to_list(),
    )
]

fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.025,
    row_heights=[0.50, 0.15, 0.12, 0.12, 0.11],
    subplot_titles=(
        "Precio, VWAP, Bollinger y Donchian",
        "Volumen",
        "RSI",
        "ATR",
        "ROC",
    ),
)

fig.add_trace(
    go.Candlestick(
        x=plot_df["plot_x"].to_list(),
        open=plot_df["open"].to_list(),
        high=plot_df["high"].to_list(),
        low=plot_df["low"].to_list(),
        close=plot_df["close"].to_list(),
        name="Precio",
    ),
    row=1,
    col=1,
)

# Precio
add_segmented_line(fig, plot_df, "vwap_daily", "VWAP", 1,  "#1030e7", width=2)

# Bandas de Bollinger
# add_segmented_line(fig, plot_df, "bb_upper_50", "BB superior", 1, "#7f7f7f", "dot")
# add_segmented_line(fig, plot_df, "bb_middle_50", "BB media", 1, "#7f7f7f")
# add_segmented_line(fig, plot_df, "bb_lower_50", "BB inferior", 1, "#7f7f7f", "dot")

# Canales de Donchian
add_segmented_line(
    fig, plot_df, "donchian_upper_20", "Donchian superior", 1, "#9467bd", "dash"
)
add_segmented_line(
    fig, plot_df, "donchian_middle_20", "Donchian medio", 1, "#9467bd", "dot"
)
add_segmented_line(
    fig, plot_df, "donchian_lower_20", "Donchian inferior", 1, "#9467bd", "dash"
)
add_segmented_line(
    fig, plot_df, "ema_50", "ema 50", 1, "#1dc0dd", "solid"
)
# add_segmented_line(
#     fig, plot_df, "ema_10", "ema 10", 1, "#ca4095", "solid"
# )

# Volumen: vela alcista rojo, vela bajista verde
fig.add_trace(
    go.Bar(
        x=plot_df["plot_x"].to_list(),
        y=plot_df["volume"].to_list(),
        marker_color=volume_colors,
        name="Volumen",
    ),
    row=2,
    col=1,
)
#agrego la linea de media y mediana volumen
# add_segmented_line(fig, plot_df, "mean_vol_100", "mean_vol_100", 2, "#922909", width=2)
# add_segmented_line(fig, plot_df, "median_vol_100", "median_vol_100", 2, "#b89112", width=2)



# Indicadores inferiores
add_segmented_line(fig, plot_df, "atr_14", "ATR 14", 4, "#1f77b4", width=2)
add_segmented_line(fig, plot_df, "rsi_14", "RSI 14", 3, "#ff7f0e", width=2)
add_segmented_line(fig, plot_df, "roc_10", "ROC 10", 5, "#690d40", width=2)
# add_segmented_line(fig, plot_df, "velocity_5_3", "velocity_5_3", 4, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "williams_r_14", "williams_r_14", 4, "#2ca02c", width=2)
# fig.update_yaxes(range=[0, -100], row=4, col=1)
# fig.add_hline(y=-80, row=4, col=1, line_dash="dot", line_color="red")
# fig.add_hline(y=-20, row=4, col=1, line_dash="dot", line_color="green")
# add_segmented_line(fig, plot_df, "acceleration_5_3_1", "acceleration_5_3_1", 5, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "rsi_3", "RSI 3", 5, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "vortex_plus_14", "Vortex 14 plus", 5, "#2ca02c", width=2)
# add_segmented_line(fig, plot_df, "vortex_minus_14", "Vortex 14 minus", 5, "#96280c", width=2)
add_segmented_line(fig, plot_df, "mean_vol_100", "Mean Vol 100", 2, "#96280c", width=2)


# Una marca de eje por inicio de cada día y ticker.
day_starts = (
    plot_df
    .group_by(GROUPS, maintain_order=True)
    .agg(pl.col("plot_x").min().alias("plot_x"))
    .sort("plot_x")
)

fig.update_xaxes(
    tickmode="array",
    tickvals=day_starts["plot_x"].to_list(),
    ticktext=[
        f"{row['ticker']}<br>{row['trading_day']}"
        for row in day_starts.to_dicts()
    ],
    row=5,
    col=1,
)

# Etiqueta centrada sobre cada tramo de ticker.
ticker_ranges = (
    plot_df
    .group_by("ticker", maintain_order=True)
    .agg(
        pl.col("plot_x").min().alias("start"),
        pl.col("plot_x").max().alias("end"),
    )
)

# for ticker_range in ticker_ranges.to_dicts():
#     fig.add_annotation(
#         x=(ticker_range["start"] + ticker_range["end"]) / 2,
#         y=1.03,
#         xref="x",
#         yref="paper",
#         text=f"<b>{ticker_range['ticker']}</b>",
#         showarrow=False,
#     )
hover_labels = {
    row["plot_x"]: (
        f"<b>{row['ticker']}</b><br>"
        f"{row['datetime'].strftime('%d/%m/%Y %H:%M:%S %Z')}"
    )
    for row in plot_df.select(
        "plot_x", "ticker", "datetime"
    ).iter_rows(named=True)
}

for trace in fig.data:
    if trace.type not in ("candlestick", "scatter", "bar"):
        continue

    labels = [hover_labels[x] for x in trace.x]

    if trace.type == "candlestick":
        trace.update(
            hovertext=[
                (
                    f"{label}<br>"
                    f"Apertura: {o:.4f}<br>"
                    f"Máximo: {h:.4f}<br>"
                    f"Mínimo: {l:.4f}<br>"
                    f"Cierre: {c:.4f}"
                )
                for label, o, h, l, c in zip(
                    labels,
                    trace.open,
                    trace.high,
                    trace.low,
                    trace.close,
                )
            ],
            hoverinfo="text",
        )
    else:
        trace.update(
            text=labels,
            hovertemplate=(
                "%{text}<br>"
                "Valor: %{y:,.4f}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
fig.update_xaxes(
    showspikes=True,
    spikesnap="cursor",
    spikemode="across",
    spikecolor="#808080",
    spikethickness=1,
)

fig.update_layout(
    xaxis_rangeslider_visible=False,
    hovermode="x",
    # hoversubplots="axis",
    height=1_300,
    spikedistance=-1,
)

fig.show()

Conversion de data para backtesting

In [ ]:
df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,atr_14,rsi_14,donchian_upper_50,donchian_lower_50,donchian_middle_50,momentum_10,mean_vol_100,ema_50,roc_10,bb_middle_50,bb_upper_50,bb_lower_50
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-10-13 04:00:00 EDT,"""TCRT""",1.0,1.01,0.9897,0.9897,2177,2026-10-13,0.996467,0.996467,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:03:00 EDT,"""TCRT""",0.9905,1.02,0.9896,0.9896,7586,2026-10-13,0.999733,0.999005,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:04:00 EDT,"""TCRT""",0.9896,1.0,0.9896,1.0,472,2026-10-13,0.996533,0.998891,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:09:00 EDT,"""TCRT""",1.03,1.05,1.03,1.05,1357,2026-10-13,1.043333,1.004094,null,null,null,null,null,null,null,null,null,null,null,null
2026-10-13 04:14:00 EDT,"""TCRT""",1.0499,1.0499,1.0499,1.0499,363,2026-10-13,1.0499,1.005484,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-10-09 19:55:00 EDT,"""VEEA""",5.04,5.05,4.99,4.99,5726,2026-10-09,5.01,5.931664,0.062907,54.661098,5.09,4.7762,4.9331,0.0354,2312.0,4.914584,0.714488,4.914584,5.080254,4.748915
2026-10-09 19:56:00 EDT,"""VEEA""",4.98,4.98,4.95,4.95,4462,2026-10-09,4.96,5.931612,0.06005,50.272795,5.09,4.7762,4.9331,-0.05,2346.11,4.915973,-1.0,4.915973,5.08031,4.751636
2026-10-09 19:57:00 EDT,"""VEEA""",4.98,4.98,4.9715,4.98,1456,2026-10-09,4.977167,5.931596,0.058764,53.300911,5.09,4.7762,4.9331,0.01,2356.4,4.918484,0.201207,4.918484,5.08357,4.753398


In [ ]:
# [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]

In [ ]:
data = df_to_plot.filter(pl.col('ticker')=="VEEA")
# .filter(pl.col('datetime').dt.date()>=date.fromisoformat('2026-10-04'))
data.columns = [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]
data = data.to_pandas().set_index('Datetime')
data

,Ticker,Open,High,Low,Close,Volume,Trading_day,Price,Vwap_daily,Atr_14,...,Donchian_upper_50,Donchian_lower_50,Donchian_middle_50,Momentum_10,Mean_vol_100,Ema_50,Roc_10,Bb_middle_50,Bb_upper_50,Bb_lower_50
Datetime,,,,,,,,,,,,,,,,,,,,,
2026-10-05 04:00:00-04:00,VEEA,4.3100,4.7406,4.1500,4.1500,78893,2026-10-05,4.346867,4.346867,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-05 04:01:00-04:00,VEEA,4.1500,4.5800,4.1500,4.1506,35747,2026-10-05,4.293533,4.330236,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-05 04:02:00-04:00,VEEA,4.1593,4.7600,4.1304,4.3900,381615,2026-10-05,4.426800,4.404493,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-05 04:03:00-04:00,VEEA,4.3900,4.4900,4.1000,4.1004,63551,2026-10-05,4.230133,4.384699,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-05 04:04:00-04:00,VEEA,4.1005,4.1100,4.0910,4.0910,13072,2026-10-05,4.097333,4.378142,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-09 19:55:00-04:00,VEEA,5.0400,5.0500,4.9900,4.9900,5726,2026-10-09,5.010000,5.931664,0.062907,...,5.09,4.7762,4.9331,0.0354,2312.00,4.914584,0.714488,4.914584,5.080254,4.748915
2026-10-09 19:56:00-04:00,VEEA,4.9800,4.9800,4.9500,4.9500,4462,2026-10-09,4.960000,5.931612,0.060050,...,5.09,4.7762,4.9331,-0.0500,2346.11,4.915973,-1.000000,4.915973,5.080310,4.751636
2026-10-09 19:57:00-04:00,VEEA,4.9800,4.9800,4.9715,4.9800,1456,2026-10-09,4.977167,5.931596,0.058764,...,5.09,4.7762,4.9331,0.0100,2356.40,4.918484,0.201207,4.918484,5.083570,4.753398


In [ ]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
from backtesting.test import SMA, GOOG


class VwapCross(Strategy):


    max_bars = 5
    valor_R=2
    # def bollinger_bands_indicator(self):
    #     return np.vstack(
    #         (
    #             self.data.Bb_upper_20,
    #             self.data.Bb_middle_20,
    #             self.data.Bb_lower_20,
    #         )
    #     )
    # def donchian_channels_indicator(self):
    #         return np.vstack(
    #             (
    #                 self.data.Donchian_upper_20	,
    #                 self.data.Donchian_lower_20,
    #                 self.data.Donchian_middle_20,
    #             )
    #         )
    
    def vwap_indicator(self):
        return self.data.Vwap_daily
    def atr_indicator(self):
            return self.data.Atr_14
    # def momentum_indicator(self):
    #         return self.data.Momentum_10
    def rsi_indicator(self):
         return self.data.Rsi_14
    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.atr = self.I(self.atr_indicator)
        # self.momentum= self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)
        self.ema50 = self.I( 
            np.asarray,
            self.data.Ema_50,
            name="EMA 50",
            overlay=True,)
        self.roc10 = self.I( 
                    np.asarray,
                    self.data.Roc_10,
                    name="ROC 10",
                    overlay=False,)


        # self.bb_upper, self.bb_middle, self.bb_lower = self.I(
        #     self.bollinger_bands_indicator,
        #     name=("BB superior", "BB media", "BB inferior"),
        #     overlay=True,
        # )
        # self.donchian_upper, self.donchian_middle, self.donchian_lower = self.I(
        #     self.donchian_channels_indicator,
        #     name=("donchian superior", "donchian media", "donchian inferior"),
        #     overlay=True,
        # )
        self.nbars=0
        # self.ma2 = self.I(SMA, price, 20)
        self.data_ant=None

    def next(self):
        # if crossover(self.data.Close, self.data.Vwap_daily) and not self.position.is_long:
        if self.data.Close[-1]> self.data.Vwap_daily[-1] and self.data.Close[-2]< self.data.Vwap_daily[-2] and not self.trades and not self.orders:
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ):
                # sl= self.data.Low[-3:].min()
                # tp =self.data.Close+ (self.data.Close-sl)*2
                sl= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl2= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl3= self.data.Close[-1]-self.data.Atr_14[-1]*2
                tp =self.data.Close[-1]+self.data.Atr_14[-1]*2
                tp2 =self.data.Close[-1]+self.data.Atr_14[-1]*3
                tp3 =self.data.Close[-1]+self.data.Atr_14[-1]*6
                self.nbars=0
                if sl>0 and tp>0:
                    # print(self.data.Close[-1],sl,tp)

                    size = int(self.valor_R/(self.data.Close[-1]-sl))
                    size2=int (size/4)
                    size3=int (size/4)
                    size1 = size-size2-size3
                    # print(size)
                    # print(self.data.Close[-1],sl,tp,size,size1,size2,size3)
                    # self.buy(sl=sl,tp=tp,size=size1)
                    # self.buy(sl=sl2,tp=tp2,size=size2)
                    # self.buy(sl=sl3,tp=tp3,size=size3)
                    if size1>0 and size2>0 and size3>0: 
                        self.buy(sl=sl, tp=tp, size=size1, tag="salida 1")
                        self.buy(sl=sl2, size=size2, tag="salida 2")
                        self.buy(sl=sl3, size=size3, tag="salida 3")

        open_price = self.data.Open[-1]
        high_price = self.data.High[-1]
        low_price = self.data.Low[-1]
        close_price = self.data.Close[-1]
        previous_open = self.data.Open[-2]
        previous_close = self.data.Close[-2]
        
        body = abs(close_price - open_price)
        upper_wick = high_price - max(open_price, close_price)
        lower_wick = min(open_price, close_price) - low_price
        upper_wick_signal = (
            body > 0
            and upper_wick > body * 3
            and lower_wick < upper_wick*1.5
        )

        bearish_engulfing_signal = (
            previous_close > previous_open      # vela anterior alcista
            and close_price < open_price        # vela actual bajista
            and open_price >= previous_close    # abre por encima del cierre previo
            and close_price <= previous_open    # cierra por debajo de la apertura previa
            and body > abs(previous_close - previous_open)
        )

        salida_1_abierta = any(
            trade.tag == "salida 1" for trade in self.trades
        )

        if not salida_1_abierta:
            for trade in self.trades:
                if trade.tag == "salida 2" and upper_wick_signal:
                    trade.close()

                if trade.tag == "salida 3" and bearish_engulfing_signal:
                    trade.close()
                # elif crossover(self.data.Close, self.data.vwap_daily):
                    # self.sell()
                # if self.position.is_long:
                #     self.nbars+=1
                #     if self.nbars>=self.max_bars:
                        
                #         self.position.close()
                self.data_ant = self.data





d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\backtesting\_plotting.py:55: UserWarning:

Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.



Loading BokehJS ...

In [ ]:
bt = Backtest(data, VwapCross, 
              commission=(0.001, 0),
            #   commission=(.001,0) 
              cash=500,
              exclusive_orders=False)
stats = bt.run(valor_R = 2)
stats
bt.plot()

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64



GridPlot(id='p1413', ...)

In [ ]:
import numpy as np
from backtesting import Backtest, Strategy


class VWAPCROSS_ROC_OUTPUT(Strategy):
    valor_R = 2.0
    comision_fija = 0.001
    debug = True

    def init(self):
        self.vwap = self.I(
            np.asarray, self.data.Vwap_daily,
            name="VWAP", overlay=True,
        )
        self.ema50 = self.I(
            np.asarray, self.data.Ema_50,
            name="EMA 50", overlay=True,
        )
        self.roc10 = self.I(
            np.asarray, self.data.Roc_10,
            name="ROC 10", overlay=False,
        )

        self._closed_at_entry = 0
        self._reported_closed = 0
        self._roc_exits = {}
        self.logs = []

        if not np.isfinite(self.valor_R) or self.valor_R <= 0:
            raise ValueError("valor_R debe ser positivo")

    def _log(self, event, message, timestamp=None):
        if timestamp is None:
            timestamp = self.data.index[-1]

        text = f"{timestamp} | {event} | {message}"
        self.logs.append(text)

        if self.debug:
            print(text)

    def report_closed_trades(self):
        for trade in self.closed_trades[self._reported_closed:]:
            reason = self._roc_exits.pop(id(trade), None)

            if reason is None:
                if trade.sl is not None and trade.exit_price <= trade.sl:
                    if np.isclose(
                        trade.sl, trade.entry_price, rtol=0, atol=1e-10
                    ):
                        reason = "Stop en breakeven (inferido)"
                    else:
                        reason = "Stop loss (inferido)"
                elif trade.tp is not None and trade.exit_price >= trade.tp:
                    reason = "Take profit (inferido)"
                else:
                    reason = "Motivo automático no determinado"

            self._log(
                "CIERRE EJECUTADO",
                f"{trade.tag} | {reason} | "
                f"entrada={trade.entry_price:.4f} | "
                f"salida={trade.exit_price:.4f} | "
                f"acciones={trade.size} | P/L={trade.pl:.4f}",
                timestamp=trade.exit_time,
            )

        self._reported_closed = len(self.closed_trades)

    def next(self):
        self.report_closed_trades()

        if self.trades:
            roc = self.roc10[-4:]
            roc_description = "ROC insuficiente o no válido"

            if len(roc) == 4 and np.isfinite(roc).all():
                roc_mean = float(np.mean(roc[:-1]))
                roc_description = (
                    f"ROC actual={roc[-1]:.4f}, "
                    f"media de los 3 anteriores={roc_mean:.4f}"
                )

                if roc[-1] < roc_mean:
                    for trade in self.trades:
                        reason = f"Salida por ROC: {roc_description}"
                        self._roc_exits[id(trade)] = reason
                        self._log(
                            "SOLICITUD DE CIERRE",
                            f"{trade.tag} | {reason}",
                        )
                        trade.close()
                    return

            first_closed = any(
                trade.tag == "salida 1"
                for trade in self.closed_trades[self._closed_at_entry:]
            )

            if first_closed:
                for trade in self.trades:
                    if trade.tag in ("salida 2", "salida 3"):
                        new_sl = max(trade.sl or 0, trade.entry_price)

                        if trade.sl != new_sl:
                            previous_sl = trade.sl
                            trade.sl = new_sl
                            self._log(
                                "BREAKEVEN",
                                f"{trade.tag} | cerró salida 1 | "
                                f"SL anterior={previous_sl} | "
                                f"nuevo SL={new_sl:.4f}",
                            )

            self._log(
                "SIN NUEVA ENTRADA",
                f"Hay {len(self.trades)} posiciones abiertas | "
                f"{roc_description}",
            )
            return

        if self.orders:
            self._log(
                "SIN ENTRADA",
                f"Hay {len(self.orders)} órdenes pendientes",
            )
            return

        if len(self.data) < 2:
            self._log("SIN ENTRADA", "Falta una vela anterior")
            return

        close = float(self.data.Close[-1])
        open_ = float(self.data.Open[-1])
        vwap = float(self.vwap[-1])
        ema = float(self.ema50[-1])
        atr = float(self.data.Atr_14[-1])
        volume = float(self.data.Volume[-1])
        volume_mean = float(self.data.Mean_vol_100[-1])
        previous_close = float(self.data.Close[-2])
        previous_vwap = float(self.vwap[-2])
        previous_ema = float(self.ema50[-2])

        values = {
            "close": close,
            "open": open_,
            "VWAP": vwap,
            "EMA50": ema,
            "ATR": atr,
            "volumen": volume,
            "media_volumen": volume_mean,
            "cierre_anterior": previous_close,
            "VWAP_anterior": previous_vwap,
        }

        invalid = [
            name for name, value in values.items()
            if not np.isfinite(value)
        ]

        if invalid:
            self._log(
                "SIN ENTRADA",
                "Datos no válidos o sin calentamiento: " + ", ".join(invalid),
            )
            return

        if atr <= 0 or close <= 0:
            self._log(
                "SIN ENTRADA",
                f"ATR o precio no positivo: ATR={atr}, close={close}",
            )
            return

        vwap_cross_up = (
            previous_close <= previous_vwap 
            and close > vwap
            
        )
        ema_cross_up = (
                    previous_close <= previous_ema
                    and close > ema
                    
                )

        checks = [
            (
                close > previous_close,
                f"El cierre no supera al anterior "
                f"({close:.4f} <= {previous_close:.4f})",
            ),
            (
                vwap_cross_up,
                f"No hay cruce alcista de VWAP "
                f"(cierre anterior={previous_close:.4f}, "
                f"VWAP anterior={previous_vwap:.4f}, "
                f"cierre={close:.4f}, VWAP={vwap:.4f})",
            ),
            (
                close > ema,
                f"El cierre no supera EMA50 "
                f"({close:.4f} <= {ema:.4f})",
            ),
            (
                volume > volume_mean,
                f"El volumen no supera su media "
                f"({volume:.0f} <= {volume_mean:.2f})",
            ),
            (
                vwap > ema,
                f"VWAP no supera EMA50 ({vwap:.4f} <= {ema:.4f})",
            ),
        ]

        # checks = [
        #     (
        #         close > previous_close,
        #         f"El cierre no supera al anterior "
        #         f"({close:.4f} <= {previous_close:.4f})",
        #     ),
        #     (
        #         ema_cross_up,
        #         f"No hay cruce alcista de Ema "
        #         f"(cierre anterior={previous_close:.4f}, "
        #         f"Ema anterior={previous_ema:.4f}, "
        #         f"cierre={close:.4f}, VWAP={ema:.4f})",
        #     ),
        #     (
        #         close > vwap,
        #         f"El cierre no supera VWAP "
        #         f"({close:.4f} <= {ema:.4f})",
        #     ),
        #     (
        #         volume > volume_mean,
        #         f"El volumen no supera su media "
        #         f"({volume:.0f} <= {volume_mean:.2f})",
        #     ),
        #     (
        #         vwap < ema,
        #         f"VWAP no supera EMA50 ({vwap:.4f} <= {ema:.4f})",
        #     ),
        #     (
        #         vwap < ema,
        #         f"VWAP no supera EMA50 ({vwap:.4f} <= {ema:.4f})",
        #     ),
        # ]
        
        failed = [reason for passed, reason in checks if not passed]

        if failed:
            self._log("SIN ENTRADA", " | ".join(failed))
            return

        sl = close - 2 * atr

        if sl <= 0:
            self._log(
                "SIN ENTRADA",
                f"El stop calculado no es positivo: SL={sl:.4f}",
            )
            return

        max_risk_size = int(self.valor_R / (2 * atr))
        max_cash_size = max(
            0,
            int((self.equity - 6 * self.comision_fija) / close),
        )
        quarter = min(max_risk_size, max_cash_size) // 4

        if quarter < 1:
            self._log(
                "SIN ENTRADA",
                f"No alcanza para repartir 4 acciones | "
                f"máximo por riesgo={max_risk_size} | "
                f"máximo por capital={max_cash_size} | ATR={atr:.4f}",
            )
            return

        self._closed_at_entry = len(self.closed_trades)
        # self._log('****ENTRADA ',
        #          f"Vwap previa: {previous_vwap}| Vwap {vwap} |",
        #          f"close previo: {previous_close}| Close {close} | {self.data.Close[-1]}")
        self.buy(
            size=2 * quarter,
            sl=sl,
            tp=close + 4 * atr,
            tag="salida 1",
        )
        self.buy(
            size=quarter,
            sl=sl,
            tp=close + 6 * atr,
            tag="salida 2",
        )
        self.buy(
            size=quarter,
            sl=sl,
            tag="salida 3",
        )

        self._log(
            "ENTRADA SOLICITADA",
            f"Todos los filtros cumplidos: cierre creciente, cruce VWAP, "
            f"cierre sobre EMA50, volumen superior a su media y VWAP sobre EMA50 | "
            f"precio de referencia={close:.4f} | "
            f"acciones={2 * quarter}/{quarter}/{quarter} | "
            f"SL={sl:.4f} | TP1={close + 4 * atr:.4f} | "
            f"TP2={close + 6 * atr:.4f}",
        )

In [ ]:
bt = Backtest(data, VWAPCROSS_ROC_OUTPUT, 
              commission=(0.001, 0),
            #   commission=(.001,0) 
              cash=500,
              exclusive_orders=False)
stats = bt.run(
    valor_R=10,
    comision_fija=0.001,
    debug=True,
)

stats["_strategy"].report_closed_trades()
# stats = bt.run(valor_R = 10)

bt.plot()

2026-10-05 04:50:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:51:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:52:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:53:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:54:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:55:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:56:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:57:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:58:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:59:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 05:00:00-04:00 | SIN ENTRADA | Datos no válidos o

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64



GridPlot(id='p1805', ...)

In [ ]:
stats

Start                     2026-10-05 04:00...
End                       2026-10-09 19:59...
Duration                      4 days 15:59:00
Exposure Time [%]                      2.0494
Equity Final [$]                    525.78143
Equity Peak [$]                     530.04243
Commissions [$]                          0.06
Return [%]                            5.15629
Buy & Hold Return [%]                15.16977
Return (Ann.) [%]                   -37.49818
Volatility (Ann.) [%]                      --
CAGR [%]                           5019.78379
Sharpe Ratio                               --
Sortino Ratio                       -12.67772
Calmar Ratio                        -20.25853
Alpha [%]                              5.0275
Beta                                  0.00849
Max. Drawdown [%]                    -1.85098
Avg. Drawdown [%]                    -1.20048
Max. Drawdown Duration        0 days 11:14:00
Avg. Drawdown Duration        0 days 04:22:00
# Trades                          

In [ ]:
stats._trades

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_VWAP,Exit_VWAP,Entry_EMA 50,Exit_EMA 50,Entry_ROC 10,Exit_ROC 10
0,11,103,105,4.7602,4.690300,4.568600,NaN,-0.770900,0.002,-0.014722,2026-10-05 05:43:00-04:00,2026-10-05 05:45:00-04:00,0 days 00:02:00,salida 3,4.752022,4.751806,4.690472,4.691651,4.869361,2.840959
1,11,103,105,4.7602,4.690300,4.568600,5.405800,-0.770900,0.002,-0.014722,2026-10-05 05:43:00-04:00,2026-10-05 05:45:00-04:00,0 days 00:02:00,salida 2,4.752022,4.751806,4.690472,4.691651,4.869361,2.840959
2,22,103,105,4.7602,4.690300,4.568600,5.196500,-1.539800,0.002,-0.014703,2026-10-05 05:43:00-04:00,2026-10-05 05:45:00-04:00,0 days 00:02:00,salida 1,4.752022,4.751806,4.690472,4.691651,4.869361,2.840959
3,44,169,171,4.8204,5.043057,4.709371,5.043057,9.794914,0.002,0.046181,2026-10-05 06:49:00-04:00,2026-10-05 06:51:00-04:00,0 days 00:02:00,salida 1,4.737662,4.742581,4.620212,4.641053,7.236842,8.107221
4,22,169,175,4.8204,4.900400,4.820400,NaN,1.758000,0.002,0.016577,2026-10-05 06:49:00-04:00,2026-10-05 06:55:00-04:00,0 days 00:06:00,salida 3,4.737662,4.750827,4.620212,4.682174,7.236842,4.274298
5,22,169,175,4.8204,4.900400,4.820400,5.154286,1.758000,0.002,0.016577,2026-10-05 06:49:00-04:00,2026-10-05 06:55:00-04:00,0 days 00:06:00,salida 2,4.737662,4.750827,4.620212,4.682174,7.236842,4.274298
6,8,356,358,5.5600,5.580000,5.241529,NaN,0.158000,0.002,0.003552,2026-10-05 09:56:00-04:00,2026-10-05 09:58:00-04:00,0 days 00:02:00,salida 3,5.281313,5.291848,5.196907,5.229483,13.163819,9.176812
7,8,356,358,5.5600,5.580000,5.241529,6.475414,0.158000,0.002,0.003552,2026-10-05 09:56:00-04:00,2026-10-05 09:58:00-04:00,0 days 00:02:00,salida 2,5.281313,5.291848,5.196907,5.229483,13.163819,9.176812
8,16,356,358,5.5600,5.580000,5.241529,6.166943,0.318000,0.002,0.003575,2026-10-05 09:56:00-04:00,2026-10-05 09:58:00-04:00,0 days 00:02:00,salida 1,5.281313,5.291848,5.196907,5.229483,13.163819,9.176812
9,14,406,407,5.3300,5.310000,5.171486,NaN,-0.282000,0.002,-0.003779,2026-10-05 10:46:00-04:00,2026-10-05 10:47:00-04:00,0 days 00:01:00,salida 3,5.294108,5.294113,5.168360,5.174303,3.006790,3.299029


In [ ]:
for message in stats["_strategy"].logs:
    print(message)

2026-10-05 04:50:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:51:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:52:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:53:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:54:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:55:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:56:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:57:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:58:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 04:59:00-04:00 | SIN ENTRADA | Datos no válidos o sin calentamiento: media_volumen
2026-10-05 05:00:00-04:00 | SIN ENTRADA | Datos no válidos o

In [ ]:
import numpy as np
import pandas as pd
from datetime import time
from backtesting import Backtest, Strategy


class ORB(Strategy):
    valor_R = 2.0
    objetivo_R = 2.0
    comision_fija = 0.001
    debug = True

    def init(self):
        if self.valor_R <= 0 or self.objetivo_R <= 0:
            raise ValueError("valor_R y objetivo_R deben ser positivos")

        index = self.data.index

        if not isinstance(index, pd.DatetimeIndex) or index.tz is None:
            raise ValueError("El índice debe tener zona horaria")

        self._horas = index.tz_convert("America/New_York")

        if not index.is_monotonic_increasing or index.has_duplicates:
            raise ValueError("El índice debe estar ordenado y sin duplicados")

        self._dia = None
        self._operado = False
        self._reportados = 0
        self.logs = []

    def _log(self, evento, mensaje, timestamp=None):
        timestamp = (
            self.data.index[-1] if timestamp is None else timestamp
        )
        texto = f"{timestamp} | {evento} | {mensaje}"
        self.logs.append(texto)

        if self.debug:
            print(texto)

    def report_closed_trades(self):
        for trade in self.closed_trades[self._reportados:]:
            self._log(
                "CIERRE",
                f"entrada={trade.entry_price:.4f} | "
                f"salida={trade.exit_price:.4f} | "
                f"acciones={trade.size} | P/L={trade.pl:.4f}",
                timestamp=trade.exit_time,
            )

        self._reportados = len(self.closed_trades)

    def next(self):
        self.report_closed_trades()

        ahora = self._horas[len(self.data) - 1]
        hora = ahora.time()

        if ahora.date() != self._dia:
            self._dia = ahora.date()
            self._operado = False

        # Con datos de 1 minuto y trade_on_close=True,
        # el cierre solicitado aquí se procesa en la barra 15:59.
        if hora >= time(15, 58):
            for order in list(self.orders):
                if not order.is_contingent:
                    order.cancel()

            if self.position:
                self.position.close()
                self._log("SALIDA", "Cierre por fin de sesión")

            return

        if hora < time(9, 45):
            return

        if self._operado or self.position or self.orders:
            return

        # Solo consulta barras ya disponibles: no usa precios futuros.
        horas = self._horas[:len(self.data)]
        apertura = (
            (horas.date == ahora.date())
            & (horas.time >= time(9, 30))
            & (horas.time < time(9, 45))
        )

        # Exige las 15 barras iniciales completas.
        esperadas = pd.date_range(
            ahora.normalize() + pd.Timedelta(hours=9, minutes=30),
            periods=15,
            freq="min",
        )

        if not horas[apertura].equals(esperadas):
            return

        maximo = float(np.max(np.asarray(self.data.High)[apertura]))
        minimo = float(np.min(np.asarray(self.data.Low)[apertura]))
        cierre = float(self.data.Close[-1])
        anterior = float(self.data.Close[-2])

        if not np.isfinite([maximo, minimo, cierre, anterior]).all():
            return

        if minimo <= 0 or maximo <= minimo:
            return

        # Primera ruptura confirmada al cierre de una vela.
        if not (anterior <= maximo < cierre):
            return

        # Consume el intento aunque no alcance el capital.
        self._operado = True

        riesgo_accion = cierre - minimo
        size_riesgo = int(self.valor_R / riesgo_accion)
        size_capital = max(
            0,
            int((self.equity - 2 * self.comision_fija) / cierre),
        )
        size = min(size_riesgo, size_capital)

        if size < 1:
            self._log("SIN ENTRADA", "Riesgo o capital insuficiente")
            return

        objetivo = cierre + self.objetivo_R * riesgo_accion

        self.buy(
            size=size,
            sl=minimo,
            tp=objetivo,
            tag="ORB",
        )

        self._log(
            "ENTRADA",
            f"rango={minimo:.4f}–{maximo:.4f} | "
            f"referencia={cierre:.4f} | "
            f"SL={minimo:.4f} | TP={objetivo:.4f} | "
            f"acciones={size} | riesgo nominal={size * riesgo_accion:.2f}",
        )

In [ ]:
# Solo sesión regular. Conserva la zona horaria del índice.
horas_ny = data.index.tz_convert("America/New_York")

data_orb = data.loc[
    (horas_ny.time >= time(9, 30))
    & (horas_ny.time < time(16, 0))
].copy()

bt_orb = Backtest(
    data_orb,
    ORB,
    cash=500,
    commission=(0.001, 0),
    trade_on_close=True,
    exclusive_orders=True,
    finalize_trades=True,
)

stats_orb = bt_orb.run(
    valor_R=2,
    objetivo_R=2,
    comision_fija=0.001,
    debug=True,
)

stats_orb["_strategy"].report_closed_trades()

display(stats_orb)
display(stats_orb["_trades"])
bt_orb.plot()

2026-10-05 09:55:00-04:00 | ENTRADA | rango=4.8900–5.4294 | referencia=5.5500 | SL=4.8900 | TP=6.8700 | acciones=3 | riesgo nominal=1.98
2026-10-05 11:33:00-04:00 | CIERRE | entrada=5.5500 | salida=4.8900 | acciones=3 | P/L=-1.9820
2026-10-09 10:17:00-04:00 | ENTRADA | rango=5.4000–6.2400 | referencia=6.4805 | SL=5.4000 | TP=8.6415 | acciones=1 | riesgo nominal=1.08
2026-10-09 13:50:00-04:00 | CIERRE | entrada=6.4805 | salida=5.4000 | acciones=1 | P/L=-1.0825


Start                     2026-10-05 09:30...
End                       2026-10-09 15:59...
Duration                      4 days 06:29:00
Exposure Time [%]                    40.33505
Equity Final [$]                     496.9355
Equity Peak [$]                      500.2987
Commissions [$]                         0.004
Return [%]                            -0.6129
Buy & Hold Return [%]                11.56778
Return (Ann.) [%]                    -42.2096
Volatility (Ann.) [%]                      --
CAGR [%]                            -40.89546
Sharpe Ratio                               --
Sortino Ratio                       -12.23286
Calmar Ratio                        -62.78962
Alpha [%]                             -0.6752
Beta                                  0.00539
Max. Drawdown [%]                    -0.67224
Avg. Drawdown [%]                    -0.34808
Max. Drawdown Duration        4 days 06:01:00
Avg. Drawdown Duration        2 days 03:02:00
# Trades                          

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag
0,3,25,123,5.5500,4.89,4.89,6.8700,-1.9820,0.002,-0.119039,2026-10-05 09:55:00-04:00,2026-10-05 11:33:00-04:00,0 days 01:38:00,ORB
1,1,433,646,6.4805,5.40,5.40,8.6415,-1.0825,0.002,-0.167040,2026-10-09 10:17:00-04:00,2026-10-09 13:50:00-04:00,0 days 03:33:00,ORB


d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning:

no explicit representation of timezones available for np.datetime64



GridPlot(id='p2136', ...)

In [ ]:
stats._equity_curve.to_csv("equity.csv",sep=";")

In [ ]:
path_salida = os.path.join(data_dir,"output","VwapCross","results.parquet")
print(path_salida)
df = pl.read_parquet(path_salida)

E:\smallcaps_data\output\VwapCross\results.parquet


In [ ]:
df.filter(pl.col("Return [%]")<0)['Return [%]'].mean()

-0.8046699447204283